# Create a flight information table via web API

In [63]:
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv, find_dotenv
import os

Retrieve cities table from MySQL

In [64]:
load_dotenv(find_dotenv())
schema = os.getenv("SQL_schema")
host = os.getenv("SQL_host")
user = os.getenv("SQL_user")
password = os.getenv("SQL_password")
port = os.getenv("SQL_port")

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [66]:
city_df_from_sql = pd.read_sql("cities", con=connection_string)
city_df_from_sql

,city_id,city,country,latitude,longitude,area_km2
0,1,Berlin,Germany,52.52000,13.40500,891.30
1,2,Hamburg,Germany,53.55000,10.00000,755.09
2,3,Frankfurt,Germany,50.11056,8.68222,248.31
3,4,Munich,Germany,48.13750,11.57500,310.71
4,5,Cologne,Germany,50.93639,6.95278,405.15
5,6,Stuttgart,Germany,48.77750,9.18000,207.33
6,7,Düsseldorf,Germany,51.22556,6.77667,217.41
7,8,Leipzig,Germany,51.34000,12.37500,297.80
8,9,Dortmund,Germany,51.51389,7.46528,280.71
9,10,Essen,Germany,51.45083,7.01306,210.34


Function to retrieve all airports associated with each city

In [ ]:
def get_airports(city_df: pd.DataFrame):
    
    # API headers
    load_dotenv(find_dotenv())
    headers = {
        "x-rapidapi-key": os.getenv("flight_api_key"),
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    # DataFrame to store results
    data_frame_rows = []

    for city_id in city_df["city_id"]:
        # Construct the querystring with the latitude and longitude
        lat = city_df.loc[city_df["city_id"] == city_id, "latitude"].item()
        lon = city_df.loc[city_df["city_id"] == city_id, "longitude"].item()
        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {"lat":lat,"lon":lon,"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)
        time.sleep(2.5) # can't do API requests too quickly
        if response.status_code == 200:
            data = response.json()
            for i in range(len(data.get("items", []))):
                airport_icao = data.get("items", [])[i].get("icao",{})
                airport_name = data.get("items", [])[i].get("name",{})

                data_frame_rows.append({
                    "city_id": city_id,
                    "airport_icao": airport_icao,
                    "airport_name": airport_name,
                    })
        
    return pd.DataFrame(data_frame_rows)

Call function to create airports_df

In [29]:
airports_df = get_airports(city_df_from_sql)

Insert airports_df into MySQL

In [89]:
airports_df.to_sql('airports',
                  if_exists='delete_rows',
                  con=connection_string,
                  index=False)

2

Function to retrieve all arrivals associated with each aiport for tomorrow

In [ ]:
def get_arriving_flights(airports_df: pd.DataFrame):
    load_dotenv(find_dotenv())
    headers = {
        "x-rapidapi-key": os.getenv("flight_api_key"),
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    querystring = {"withLeg":"true","direction":"Arrival","withCancelled":"false","withCodeshared":"false",
                        "withCargo":"false","withPrivate":"false","withLocation":"false"}

    # fetch start and end date of tomorrow and put them in 2x 12h windows
    today_midnight = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    tomorrow = datetime.now() + timedelta(days=1)
    windows = [
        (tomorrow.strftime("%Y-%m-%dT00:00"), tomorrow.strftime("%Y-%m-%dT11:59")),
        (tomorrow.strftime("%Y-%m-%dT12:00"), (tomorrow + timedelta(days=1)).strftime("%Y-%m-%dT23:59")),
    ]

    data_frame_rows = []

    for icao in airports_df["airport_icao"]:
        for start, end in windows:
            time.sleep(2.5) # can't do API requests too quickly
            # Construct the URL with the airport infos and time windows
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{start}/{end}"
            response = requests.get(url, headers=headers, params=querystring)
            if response.status_code == 200:
                data = response.json()
                for i in range(len(data.get("arrivals", []))):
                    flight_num = data.get("arrivals", [])[i].get("number", None)
                    departure_icao = data.get("arrivals", [])[i].get("departure", {}).get("airport",{}).get("icao", None)
                    arrival_time = data.get("arrivals", [])[i].get("arrival", {}).get("scheduledTime",{}).get("local", None)
                    # data retrieval timestamp
                    data_retrieval_time = pd.Timestamp.now(tz='Europe/Berlin').tz_localize(None)

                    data_frame_rows.append({
                        "flight_num": flight_num,
                        "departure_icao": departure_icao,
                        "arrival_icao": icao,
                        "arrival_time": pd.to_datetime(arrival_time).tz_localize(None),
                        "data_retrieval_time": data_retrieval_time
                        })

    return pd.DataFrame(data_frame_rows)

Call flights function

In [108]:
flights_df = get_arriving_flights(airports_df)
flights_df

,flight_num,departure_icao,arrival_icao,arrival_time,data_retrieval_time
0,SR 149,OLBA,EDDB,2026-05-29 06:25:00,2026-05-28 13:00:26.714121
1,HU 489,ZBAA,EDDB,2026-05-29 06:45:00,2026-05-28 13:00:26.718693
2,XQ 1766,LTAJ,EDDB,2026-05-29 06:45:00,2026-05-28 13:00:26.719311
3,W4 3109,LROP,EDDB,2026-05-29 07:05:00,2026-05-28 13:00:26.719801
4,W6 7037,LZIB,EDDB,2026-05-29 07:15:00,2026-05-28 13:00:26.720293
...,...,...,...,...,...
284,FR 5418,EIDW,EDDB,2026-05-29 23:00:00,2026-05-28 13:00:29.641379
285,EW 8231,EFHK,EDDB,2026-05-29 23:00:00,2026-05-28 13:00:29.641899
286,SR 7359,LBBG,EDDB,2026-05-29 23:05:00,2026-05-28 13:00:29.642485
287,EW 8671,LGKO,EDDB,2026-05-29 23:05:00,2026-05-28 13:00:29.642993


Insert flights_df into MySQL

In [109]:
flights_df.to_sql('flights',
                  if_exists='append',
                  con=connection_string,
                  index=False)

289

Wrap everything in functions, so it is only necessary to call one function in the end

In [ ]:
"""
AeroDataBox flight data fetcher.

Finds airports within 50 km of each city stored in the MySQL 'cities' table,
then fetches tomorrow's arriving flights for each airport across two 12-hour
windows and persists everything to 'airports' and 'flights'.
"""

import pandas as pd
import requests
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv, find_dotenv
import os

def create_connection_string():
    """Build a SQLAlchemy connection string from environment variables.

    Reads SQL_schema, SQL_host, SQL_user, SQL_password, and SQL_port from
    the nearest .env file and returns a mysql+pymysql:// URL.
    """
    load_dotenv(find_dotenv())
    schema = os.getenv("SQL_schema")
    host = os.getenv("SQL_host")
    user = os.getenv("SQL_user")
    password = os.getenv("SQL_password")
    port = os.getenv("SQL_port")

    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

def fetch_cities_data(connection_string: str):
    """Load the full 'cities' table, including city_id and coordinates.

    Args:
        connection_string: SQLAlchemy connection URL.

    Returns:
        DataFrame mirroring the 'cities' table schema.
    """
    return pd.read_sql("cities", con=connection_string)

def get_airports(city_df: pd.DataFrame):
    """Find airports within 50 km of each city via the AeroDataBox API.

    Only airports that have active flight information are returned
    (withFlightInfoOnly=true). A 2.5-second sleep is inserted between
    requests to stay within the API's rate limit.

    Args:
        city_df: DataFrame with at least city_id, latitude, and longitude
                 columns, so the result of fetch_cities_data().

    Returns:
        DataFrame with columns: city_id, airport_icao, airport_name.
    """
    # API headers
    load_dotenv(find_dotenv())
    headers = {
        "x-rapidapi-key": os.getenv("flight_api_key"),
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    data_frame_rows = []

    for city_id in city_df["city_id"]:
        # Construct the querystring with the latitude and longitude
        lat = city_df.loc[city_df["city_id"] == city_id, "latitude"].item()
        lon = city_df.loc[city_df["city_id"] == city_id, "longitude"].item()
        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {"lat":lat,"lon":lon,"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)
        time.sleep(2.5) # can't do API requests too quickly
        if response.status_code == 200:
            data = response.json()
            for i in range(len(data.get("items", []))):
                airport_icao = data.get("items", [])[i].get("icao",{})
                airport_name = data.get("items", [])[i].get("name",{})

                data_frame_rows.append({
                    "city_id": city_id,
                    "airport_icao": airport_icao,
                    "airport_name": airport_name,
                    })
        
    return pd.DataFrame(data_frame_rows)

def store_airports_data(airports_df: pd.DataFrame, connection_string: str):
    """Replace all rows in the 'airports' table with the new airport data.

    Args:
        airports_df: DataFrame produced by get_airports().
        connection_string: SQLAlchemy connection URL.
    """
    airports_df.to_sql('airports',
                  if_exists='delete_rows',
                  con=connection_string,
                  index=False)

def get_arriving_flights(airports_df: pd.DataFrame):
    """Fetch tomorrow's arriving flights for every airport in airports_df.

    Tomorrow's schedule is split into two 12-hour windows (00:00 to 11:59 and
    12:00 to 23:59) because the AeroDataBox endpoint caps each query at 12 hours.
    Cancelled, codeshared, cargo, and private flights are excluded.
    A 2.5-second sleep is inserted before each request to respect rate limits.

    Args:
        airports_df: DataFrame with at least an airport_icao column —
                     typically the result of get_airports().

    Returns:
        DataFrame with columns: flight_num, departure_icao, arrival_icao,
        arrival_time, data_retrieval_time.
    """
    load_dotenv(find_dotenv())
    headers = {
        "x-rapidapi-key": os.getenv("flight_api_key"),
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    querystring = {"withLeg":"true","direction":"Arrival","withCancelled":"false","withCodeshared":"false",
                        "withCargo":"false","withPrivate":"false","withLocation":"false"}

    # fetch start and end date of tomorrow and put them in 2x 12h windows
    today_midnight = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    tomorrow = datetime.now() + timedelta(days=1)
    windows = [
        (tomorrow.strftime("%Y-%m-%dT00:00"), tomorrow.strftime("%Y-%m-%dT11:59")),
        (tomorrow.strftime("%Y-%m-%dT12:00"), (tomorrow + timedelta(days=1)).strftime("%Y-%m-%dT23:59")),
    ]

    data_frame_rows = []

    for icao in airports_df["airport_icao"]:
        for start, end in windows:
            time.sleep(2.5) # can't do API requests too quickly
            # Construct the URL with the airport infos and time windows
            url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{start}/{end}"
            response = requests.get(url, headers=headers, params=querystring)
            if response.status_code == 200:
                data = response.json()
                for i in range(len(data.get("arrivals", []))):
                    flight_num = data.get("arrivals", [])[i].get("number", None)
                    departure_icao = data.get("arrivals", [])[i].get("departure", {}).get("airport",{}).get("icao", None)
                    arrival_time = data.get("arrivals", [])[i].get("arrival", {}).get("scheduledTime",{}).get("local", None)
                    # data retrieval timestamp
                    data_retrieval_time = pd.Timestamp.now(tz='Europe/Berlin').tz_localize(None)

                    data_frame_rows.append({
                        "flight_num": flight_num,
                        "departure_icao": departure_icao,
                        "arrival_icao": icao,
                        "arrival_time": pd.to_datetime(arrival_time).tz_localize(None),
                        "data_retrieval_time": data_retrieval_time
                        })

    return pd.DataFrame(data_frame_rows)

def store_flights_data(flights_df: pd.DataFrame, connection_string: str):
    """Append arriving-flight records to the 'flights' table.

    Args:
        flights_df: DataFrame produced by get_arriving_flights().
        connection_string: SQLAlchemy connection URL.
    """
    flights_df.to_sql('flights',
                  if_exists='append',
                  con=connection_string,
                  index=False)

def create_airports_flights_tables():
    """End-to-end pipeline: discover airports and persist tomorrow's flights.

    Steps:
        1. Build the DB connection string from .env variables.
        2. Load city coordinates from the 'cities' table.
        3. Fetch nearby airports from AeroDataBox and replace 'airports'.
        4. Fetch tomorrow's arrivals for each airport and append to 'flights'.
    """
    connection_string = create_connection_string()
    cities_df = fetch_cities_data(connection_string)
    airports_df = get_airports(cities_df)
    store_airports_data(airports_df, connection_string)
    flights_df = get_arriving_flights(airports_df)
    store_flights_data(flights_df, connection_string)


In [112]:
create_airports_flights_tables()